# पाठ ०८ - बहु-एजेन्ट डिजाइन ढाँचा


## सेटअप


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## किन बहु-एजेन्ट प्रणालीहरू?

वास्तविक जीवनका कार्यहरू जस्तै यात्रा योजना बनाउन विभिन्न प्रकारको विशेषज्ञता आवश्यक पर्छ — रसद, स्थानीय ज्ञान, बजेटिङ, र थप। सबै कुरा एकल एजेन्टले गराउन खोज्दा यो छिटो जटिल हुन्छ।

बहु-एजेन्ट प्रणालीहरूले यसलाई **विशेषज्ञता** मार्फत समाधान गर्छन्: प्रत्येक एजेन्टले एक क्षेत्रको विशेषज्ञतामा ध्यान केन्द्रित गर्छ, जसले सामान्यज्ञ भन्दा उच्च गुणस्तरका परिणामहरू उत्पादन गर्छ। यसले **स्केलेबिलिटी** पनि सुधार गर्छ — तपाईं नयाँ एजेन्ट थप्न सक्नुहुन्छ (जस्तै, उडान विशेषज्ञ, रेस्टुरेन्ट समिक्षक) बिना हालको कार्यप्रवाहलाई फेरि लेख्नुपर्दा। एजेन्टहरूले संरचित पाइपलाइन मार्फत एक अर्कालाई सन्दर्भ पास गर्दै जोडिन्छन्।


## विशेषज्ञ एजेन्टहरू बनाउन


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## एउटा अनुक्रमिक कार्यप्रवाह निर्माण गर्दै

`WorkflowBuilder` ले तपाईंलाई एजेन्टहरूलाई निर्देशित ग्राफमा जडान गर्न दिन्छ। यहाँ हामी सरल दुई-चरण पाइपलाइन बनाउँछौं: **TravelPlanner** यात्राको योजनालाई तयार गर्छ, त्यसपछि **TravelConcierge** यसलाई समीक्षा र सुधार गर्दछ।


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## वर्कफ्लोमा थप एजेन्टहरू थप्दै

बहु-एजेन्ट ढाँचाको सबैभन्दा ठूलो फाइदाहरू मध्ये एक हो यसको विस्तार गर्न सजिलो हुनु। तल हामीले एक **BudgetReviewer** एजेन्ट थपेका छौं जसले योजनालाई यात्रुहरूको बजेटसँग तुलना गर्छ, लागत सीमा भन्दा बढी हुन सक्ने वस्तुहरूलाई झण्डा लगाउँछ, र पैसाको बचत गर्ने विकल्पहरू सुझाव दिन्छ। वर्कफ्लो अब तीन एजेन्टहरूलाई क्रमशः चलाउँछ:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## सारांश

यस पाठमा तपाईंले सिक्नुभयो कसरी:

1. **विशेषज्ञ एजेन्टहरू सिर्जना गर्ने** — प्रत्येकलाई केन्द्रित भूमिका (योजना बनाउने, कन्सर्जी, बजेट समीक्षा) दिइएको।
2. **एजेन्टहरूलाई अनुक्रमिक वर्कफ्लोमा जोड्ने** `WorkflowBuilder` र `add_edge` प्रयोग गरी।
3. **बहु-एजेन्ट पाइपलाइनबाट आउटपुट स्ट्रिम गर्ने**, कुन एजेन्ट बोल्दैछ ट्र्याक गर्दै।
4. **वर्कफ्लो विस्तार गर्ने** चेनमा नयाँ एजेन्टहरू थपेर पुराना एजेन्टहरूलाई बिना परिवर्तन।

बहु-एजेन्ट डिजाइन ढाँचाले प्रत्येक एजेन्टलाई सरल राख्छ तर कुनै एकल एजेन्टले मात्र प्राप्त गर्न सक्ने भन्दा धनी र बढी समीक्षा गरिएको नतिजा उत्पादन गर्छ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
